# StructuralTime-Core QuickStart Notebook

Welcome to the interactive demonstration of **`structural-time-core`**, a Python library designed to implement the Core Structural Time Ontology and Dynamics framework. 

This notebook provides a quick, interactive way to explore the core components without installing anything locally. You will run:
1. **Installation** directly from GitHub.
2. **Level A (Ontology):** Logical compatibility checking under the *Asymmetry Conjecture*.
3. **Level B (Dynamics):** Visualizing the Quartic Potential Well & simulating gradient flow trajectories using Runge-Kutta 4 (RK4).
4. **Analytics (Regimes):** Unsupervised clustering of states using the `HybridRegimeClustering` model.

## 1. Installation

First, we will install the library directly from the GitHub repository.

In [ ]:
!pip install git+https://github.com/dhammawatthumpra-coder/structural-time-core.git
!pip install matplotlib numpy scikit-learn

## 2. Level A: Logical Compatibility & Asymmetry Conjecture

According to the **Asymmetry Conjecture**, complex systems reject perfectly symmetric operators. Let's see how `LogicalCompatibilityChecker` flags symmetric/flat trajectories under high complexity environments.

In [ ]:
import numpy as np
from structural_time_core.ontology import LogicalCompatibilityChecker

# Create checker with complexity threshold = 0.6
checker = LogicalCompatibilityChecker(complexity_threshold=0.6)

# A symmetric/flat trajectory (zero rate of change)
flat_K = np.array([0.5, 0.5, 0.5, 0.5])

# An asymmetric/changing trajectory
changing_K = np.array([0.1, 0.3, 0.6, 0.9])

print("Under low complexity (0.4):")
print("  - Flat trajectory compatible:", checker.is_compatible(flat_K, system_complexity=0.4))

print("\nUnder high complexity (0.8):")
print("  - Flat trajectory compatible:", checker.is_compatible(flat_K, system_complexity=0.8))
print("  - Changing trajectory compatible:", checker.is_compatible(changing_K, system_complexity=0.8))

## 3. Level B: Dynamics & Quartic Potential Well

Now we'll define a double-well potential of the form:
$$\mathcal{F}(K) = a K^4 + b K^3 + c K^2 + d K$$
And compute critical equilibrium points where $d\mathcal{F}/dK = 0$.

In [ ]:
import matplotlib.pyplot as plt
from structural_time_core.dynamics import QuarticPotentialSolver

# Potential curve: a=1.0, b=0.0, c=-2.0, d=0.0 (symmetric double well)
potential = QuarticPotentialSolver(a=1.0, b=0.0, c=-2.0, d=0.0)

# Find critical points
equilibria = potential.find_equilibria()
print("Equilibria critical points:")
for eq in equilibria:
    print(f"  - K = {eq['K']:5.2f} | Type: {eq['type']:8s} | F(K) = {eq['F']:5.2f}")

# Plot the potential curve
K_range = np.linspace(-1.5, 1.5, 200)
F_vals = [potential.compute_F(k) for k in K_range]

plt.figure(figsize=(8, 5))
plt.plot(K_range, F_vals, label=r'$\mathcal{F}(K)$', color='#4f46e5', linewidth=2)

# Plot stable/unstable markers
for eq in equilibria:
    color = '#10b981' if eq['type'] == 'stable' else '#ef4444'
    plt.scatter(eq['K'], eq['F'], color=color, s=100, zorder=5, 
                label=f"{eq['type'].capitalize()} ({eq['K']:.1f})")

plt.title("Quartic Double-Well Potential", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("K state value", fontsize=12)
plt.ylabel("Free Energy F(K)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

### 3.2 Gradient Flow Simulation (RK4)

Next, we simulate the trajectory of state $K$ as it flows down the potential gradient towards a stable equilibrium point, using the 4th-order Runge-Kutta (RK4) integrator, and calculate the experienced temporal density $T(K)$.

In [ ]:
from structural_time_core.dynamics import GradientFlowIntegrator, BoundedTemporalDensityCalculator

integrator = GradientFlowIntegrator(gamma=0.05, dt=0.02)
td_calc = BoundedTemporalDensityCalculator(alpha=1.5)

# Starting near the unstable peak at K = 0.1
K_current = 0.1
trajectory = []
t_ops_vals = []

for step in range(120):
    # Calculate current gradient dF/dK
    dF_dK = potential.compute_dF_dK(K_current)
    
    # Take RK4 step
    K_next = integrator.step_rk4(K_current, potential.compute_dF_dK, u=0.0)
    velocity = abs(K_next - K_current) / integrator.dt
    
    # Distance to closest stable equilibrium (which are at -1.0 and 1.0)
    dist = min(abs(K_current - (-1.0)), abs(K_current - 1.0))
    T_ops = td_calc.compute_T_ops(velocity, dist)
    
    trajectory.append(K_current)
    t_ops_vals.append(T_ops)
    
    K_current = K_next

# Plotting the trajectory and experienced time
steps = np.arange(len(trajectory))
fig, ax1 = plt.subplots(figsize=(10, 5))

color = '#0284c7'
ax1.set_xlabel('Simulation Step', fontsize=12)
ax1.set_ylabel('K-State Value', color=color, fontsize=12)
ax1.plot(steps, trajectory, color=color, linewidth=2.5, label='K-State Trajectory')
ax1.tick_params(axis='y', labelcolor=color)
ax1.axhline(1.0, color='#10b981', linestyle='--', alpha=0.7, label='Stable Equilibrium (K=1.0)')

ax2 = ax1.twinx()
color = '#f43f5e'
ax2.set_ylabel('Experienced Time Density T(K)', color=color, fontsize=12)
ax2.plot(steps, t_ops_vals, color=color, linewidth=2, linestyle=':', label='Time Density T(K)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title("RK4 Trajectory Integration & Experienced Time Density", fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 4. Analytics: Hybrid Regime Clustering

The library categorizes states into one of 5 distinct behavioral regimes:
* **Active:** Dynamic change, moving towards a goal.
* **Critical:** Transition boundaries, high susceptibility.
* **Turbulent:** Chaotic, unstable high-energy states.
* **Decayed:** Exhausted states with high structural loss/decay (high $\gamma$).
* **Frozen:** Rest/stable states with low energy, velocity, and decay (low $\gamma$).

Let's observe how `HybridRegimeClustering` groups snapshots in 3D phase space.

In [ ]:
from structural_time_core.analytics import HybridRegimeClustering

clustering = HybridRegimeClustering()

# Snapshots format: [E_K (Energy), dK_dt (Velocity), gamma (Decay Rate)]
test_data = np.array([
    [0.85, 0.90, 0.15],  # High energy, high velocity -> Turbulent
    [0.30, 0.45, 0.08],  # Moderate energy, moderate velocity -> Active
    [0.15, 0.02, 0.65],  # Low energy, low velocity, high decay -> Decayed
    [0.08, 0.01, 0.10],  # Low energy, low velocity, low decay -> Frozen
    [0.55, 0.25, 0.20],  # Middle transition -> Critical
])

predicted_regimes = clustering.fit_predict_regimes(test_data)

print("Regime Clustering Classification Output:")
for i, sample in enumerate(test_data):
    print(f"  - Sample {i+1} [E={sample[0]:.2f}, V={sample[1]:.2f}, G={sample[2]:.2f}] -> Mapped Regime: {predicted_regimes[i]}")

## 5. Next Steps

To learn more, check out:
- [GitHub Repository](https://github.com/dhammawatthumpra-coder/structural-time-core)
- [API Documentation](https://dhammawatthumpra-coder.github.io/structural-time-core/)